# 11 — Hybrid Text + Vision Retrieval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/11_hybrid_text_plus_vision_retrieval.ipynb)


## Learning Objectives

By the end of this notebook you will be able to:

- Explain why **text-only retrieval fails** on visually-rich documents
- Build a **hybrid retrieval pipeline** that fuses text embeddings with vision embeddings
- Implement **reciprocal rank fusion (RRF)** to merge ranked lists from different modalities
- Evaluate retrieval quality with **Recall@k** and **MRR** across text-only, vision-only, and hybrid settings
- Design **fallback strategies** when one modality is missing or low-confidence


## Why Hybrid Retrieval?

Real-world documents are **multimodal by nature**. A financial report contains tables, charts, and prose. A product manual has diagrams alongside text. A slide deck communicates through layout, not sentences.

Text-only retrieval (dense or sparse) works well when the answer lives in prose. But it fails when:

1. **The answer is in a chart or table** — OCR might extract the text, but loses spatial relationships
2. **Layout carries meaning** — a two-column layout, a sidebar, a callout box
3. **The query is about visual content** — "show me the revenue trend" needs a chart, not a paragraph

Vision-only retrieval (like ColPali) treats pages as images and matches visual patterns. It excels at layout-heavy queries but struggles with precise text matching.

**Hybrid retrieval** combines both: text embeddings for semantic precision, vision embeddings for visual understanding. The key insight is that these modalities are **complementary**, not competing.


## The Hybrid Architecture

`
Query ─┬─→ Text Encoder ──→ Dense Index ──→ Ranked List A ─┐
       │                                                     ├─→ RRF ──→ Final Ranking
       └─→ Vision Encoder ─→ ColPali Index ─→ Ranked List B ─┘
`

The architecture has three stages:

1. **Dual encoding**: The query is encoded by both a text encoder (e.g., sentence-transformers) and a vision-language encoder (e.g., ColPali)
2. **Independent retrieval**: Each encoder retrieves its top-k candidates from its own index
3. **Fusion**: Reciprocal Rank Fusion merges the two ranked lists into a single ranking

### Reciprocal Rank Fusion (RRF)

RRF is elegantly simple. For each document appearing in any ranked list, its fused score is:

`
RRF_score(d) = sum(1 / (k + rank_i(d))) for each list i where d appears
`

where k is a constant (typically 60) that prevents high-ranked documents from dominating. Documents appearing in **both** lists get boosted; documents in only one list still contribute.


In [ ]:
# ── Setup ──────────────────────────────────────────────────────
!pip install -q transformers torch sentence-transformers faiss-cpu \
    pillow bitsandbytes accelerate numpy


In [ ]:
import json, numpy as np, textwrap, hashlib
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# ── Synthetic Document Corpus ──────────────────────────────────
# We create pages that have BOTH text content and visual layout
DOCUMENTS = [
    {"id": "doc_1", "text": "Q3 revenue grew 15% year-over-year to .2B, driven by cloud services.",
     "has_chart": True, "topic": "finance"},
    {"id": "doc_2", "text": "The neural network architecture uses 12 transformer layers with 768 hidden dimensions.",
     "has_chart": False, "topic": "technical"},
    {"id": "doc_3", "text": "Employee satisfaction scores improved from 72% to 85% after the new policy.",
     "has_chart": True, "topic": "hr"},
    {"id": "doc_4", "text": "The API handles 10,000 requests per second with p99 latency under 50ms.",
     "has_chart": True, "topic": "technical"},
    {"id": "doc_5", "text": "Marketing spend allocation: 40% digital, 30% events, 20% content, 10% print.",
     "has_chart": True, "topic": "marketing"},
    {"id": "doc_6", "text": "The model was fine-tuned on 50K examples using LoRA with rank 16 and alpha 32.",
     "has_chart": False, "topic": "technical"},
    {"id": "doc_7", "text": "Customer churn decreased by 23% following the onboarding redesign in Q2.",
     "has_chart": True, "topic": "product"},
    {"id": "doc_8", "text": "The retrieval pipeline uses a two-stage approach: BM25 pre-filtering then dense re-ranking.",
     "has_chart": False, "topic": "technical"},
]

print(f"Corpus: {len(DOCUMENTS)} documents, {sum(1 for d in DOCUMENTS if d['has_chart'])} with charts")


## Building the Text Retrieval Branch

The text branch uses a standard dense retriever. We encode document text with a sentence-transformer and build a FAISS index for fast approximate nearest-neighbor search.

For production systems, you would use a more powerful model like ge-large-en-v1.5 or GTE-large. Here we use ll-MiniLM-L6-v2 for speed on Colab.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# ── Text Encoder ──────────────────────────────────────────────
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all document texts
doc_texts = [d['text'] for d in DOCUMENTS]
text_embeddings = text_model.encode(doc_texts, normalize_embeddings=True)

# Build FAISS index
dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatIP(dim)  # Inner product = cosine sim for normalized vecs
text_index.add(text_embeddings.astype('float32'))

print(f"Text index: {text_index.ntotal} vectors, dim={dim}")


In [ ]:
def text_retrieve(query: str, k: int = 5) -> list[dict]:
    """Retrieve top-k documents using text embeddings."""
    q_emb = text_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = text_index.search(q_emb, k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            'doc_id': DOCUMENTS[idx]['id'],
            'score': float(score),
            'rank': rank + 1,
            'text': DOCUMENTS[idx]['text'][:80],
        })
    return results

# Test text retrieval
results = text_retrieve("What was the revenue growth?")
for r in results[:3]:
    print(f"  Rank {r['rank']}: {r['doc_id']} (score={r['score']:.3f}) — {r['text']}")


## Building the Vision Retrieval Branch

The vision branch treats each document as a rendered page image. In a real system, you would use ColPali or a similar vision-language model to encode page screenshots.

Here we simulate vision embeddings by rendering documents as images and extracting simple visual features. The key concept is the same: **pages with similar visual layouts cluster together in embedding space**, and queries about visual content (charts, tables, diagrams) find the right pages.

### Why Vision Retrieval Complements Text

Consider the query: *"show me the revenue trend"*

- **Text retrieval** finds documents mentioning "revenue" — good!
- **Vision retrieval** finds documents containing charts — also good!
- **Hybrid** promotes documents that mention revenue AND contain a chart — **best!**


In [ ]:
def render_document_as_image(doc: dict, size=(200, 100)) -> Image.Image:
    """Render a document as a synthetic page image."""
    img = Image.new('RGB', size, 'white')
    draw = ImageDraw.Draw(img)
    # Draw text content
    draw.text((5, 5), doc['text'][:50], fill='black')
    # Draw chart placeholder if document has a chart
    if doc.get('has_chart'):
        x0, y0 = 10, 50
        for i in range(5):
            h = np.random.randint(10, 40)
            draw.rectangle([x0 + i*35, y0 + 40 - h, x0 + i*35 + 25, y0 + 40], fill='steelblue')
    return img

def vision_embed(img: Image.Image) -> np.ndarray:
    """Simulate a vision embedding from a page image.
    In production, use ColPali or SigLIP for real embeddings."""
    arr = np.array(img.resize((64, 64))).flatten().astype('float32')
    # Reduce to manageable dim and normalize
    emb = arr[:384]  # match text dim for simplicity
    emb = emb / (np.linalg.norm(emb) + 1e-8)
    return emb

# Build vision index
vision_embeddings = []
for doc in DOCUMENTS:
    img = render_document_as_image(doc)
    emb = vision_embed(img)
    vision_embeddings.append(emb)

vision_embeddings = np.stack(vision_embeddings)
vision_index = faiss.IndexFlatIP(vision_embeddings.shape[1])
vision_index.add(vision_embeddings)

print(f"Vision index: {vision_index.ntotal} vectors, dim={vision_embeddings.shape[1]}")


In [ ]:
def vision_retrieve(query: str, k: int = 5) -> list[dict]:
    """Retrieve documents using vision similarity.
    Simulates: encode query as image-search, find visually similar pages."""
    # In production: use ColPali to encode query into vision space
    # Here: we use a simple heuristic — boost documents with charts for visual queries
    q_emb = text_model.encode([query], normalize_embeddings=True).astype('float32')
    # Mix text signal with vision bias
    q_vision = q_emb[:, :384]
    q_vision = q_vision / (np.linalg.norm(q_vision) + 1e-8)
    scores, indices = vision_index.search(q_vision, k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            'doc_id': DOCUMENTS[idx]['id'],
            'score': float(score),
            'rank': rank + 1,
            'has_chart': DOCUMENTS[idx]['has_chart'],
        })
    return results

results = vision_retrieve("show me the revenue chart")
for r in results[:3]:
    print(f"  Rank {r['rank']}: {r['doc_id']} (score={r['score']:.3f}, chart={r['has_chart']})")


## Reciprocal Rank Fusion

Now we implement RRF to fuse the two ranked lists. The algorithm is beautifully simple:

1. For each document in either list, compute 1 / (k + rank) where k=60
2. If a document appears in both lists, **sum** its scores
3. Sort by fused score descending

The constant k=60 was chosen by the original RRF paper (Cormack et al., 2009) to balance the contribution of high-ranked vs. lower-ranked documents. Higher k values make the fusion more egalitarian.


In [ ]:
def reciprocal_rank_fusion(
    *ranked_lists: list[dict],
    k: int = 60
) -> list[dict]:
    """Fuse multiple ranked lists using RRF.
    
    Each ranked list is a list of dicts with 'doc_id' and 'rank' keys.
    Returns a single fused ranked list.
    """
    fused_scores = {}  # doc_id -> cumulative RRF score
    doc_info = {}      # doc_id -> metadata from first appearance
    
    for list_idx, results in enumerate(ranked_lists):
        for item in results:
            doc_id = item['doc_id']
            rrf_score = 1.0 / (k + item['rank'])
            fused_scores[doc_id] = fused_scores.get(doc_id, 0) + rrf_score
            if doc_id not in doc_info:
                doc_info[doc_id] = item
    
    # Sort by fused score
    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for rank, (doc_id, score) in enumerate(ranked, 1):
        entry = {**doc_info[doc_id], 'rrf_score': score, 'rank': rank}
        results.append(entry)
    
    return results

print("RRF function ready.")


## Putting It Together: Hybrid Search

Now we combine both retrieval branches with RRF. The hybrid pipeline:

1. Takes a query
2. Runs text retrieval (top-k)
3. Runs vision retrieval (top-k)
4. Fuses results with RRF
5. Returns the final ranking


In [ ]:
def hybrid_retrieve(query: str, k: int = 5) -> dict:
    """Run hybrid text + vision retrieval with RRF fusion."""
    text_results = text_retrieve(query, k=k)
    vision_results = vision_retrieve(query, k=k)
    fused = reciprocal_rank_fusion(text_results, vision_results, k=60)
    return {
        'text_only': text_results,
        'vision_only': vision_results,
        'hybrid': fused[:k],
    }

# ── Compare All Three Approaches ────────────────────────────────
queries = [
    "What was the quarterly revenue?",
    "Show me the performance chart",
    "How is the transformer architecture structured?",
    "What is the customer churn rate?",
]

for q in queries:
    r = hybrid_retrieve(q, k=3)
    print(f"\nQuery: {q}")
    print(f"  Text:   {[x['doc_id'] for x in r['text_only'][:3]]}")
    print(f"  Vision: {[x['doc_id'] for x in r['vision_only'][:3]]}")
    print(f"  Hybrid: {[x['doc_id'] for x in r['hybrid'][:3]]}")


## Evaluating Hybrid vs. Single-Modality

How do we know hybrid is actually better? We need ground truth relevance labels and standard IR metrics.

### Key Metrics

- **Recall@k**: What fraction of relevant documents appear in the top-k results?
- **MRR (Mean Reciprocal Rank)**: How high is the first relevant document?
- **NDCG**: Normalized discounted cumulative gain — rewards relevant docs ranked higher

The hypothesis: hybrid retrieval should have **higher recall** (finds relevant docs from both modalities) and **higher MRR** (the best doc surfaces faster when both signals agree).


In [ ]:
def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    """Fraction of relevant docs found in top-k."""
    top_k = set(retrieved[:k])
    return len(top_k & relevant) / len(relevant) if relevant else 0.0

def mrr(retrieved: list[str], relevant: set[str]) -> float:
    """Mean Reciprocal Rank."""
    for i, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0

# Ground truth: for each query, which documents are relevant
ground_truth = {
    "What was the quarterly revenue?": {"doc_1"},
    "Show me the performance chart": {"doc_1", "doc_3", "doc_4", "doc_7"},
    "How is the transformer architecture structured?": {"doc_2", "doc_6"},
    "What is the customer churn rate?": {"doc_7"},
}

# Evaluate all three approaches
for mode in ['text_only', 'vision_only', 'hybrid']:
    recalls, mrrs = [], []
    for q, relevant in ground_truth.items():
        r = hybrid_retrieve(q, k=5)
        ids = [x['doc_id'] for x in r[mode]]
        recalls.append(recall_at_k(ids, relevant, k=3))
        mrrs.append(mrr(ids, relevant))
    print(f"{mode:<13} Recall@3={np.mean(recalls):.3f}  MRR={np.mean(mrrs):.3f}")


## Tuning the Fusion

Hybrid retrieval has several knobs to tune:

1. **RRF k parameter**: Higher k → more egalitarian fusion. Lower k → top-ranked results dominate.
2. **Per-modality weights**: Instead of equal contribution, weight text 0.7 and vision 0.3 (or vice versa).
3. **Retrieval depth**: Retrieve more candidates from each branch before fusion.
4. **Query routing**: Use a classifier to decide whether a query needs vision retrieval at all.

### Weighted RRF

A simple extension: multiply each modality's RRF contribution by a weight.


In [ ]:
def weighted_rrf(
    ranked_lists: list[list[dict]],
    weights: list[float],
    k: int = 60
) -> list[dict]:
    """Weighted RRF — each modality gets a different weight."""
    fused_scores = {}
    doc_info = {}
    
    for w, results in zip(weights, ranked_lists):
        for item in results:
            doc_id = item['doc_id']
            rrf_score = w / (k + item['rank'])
            fused_scores[doc_id] = fused_scores.get(doc_id, 0) + rrf_score
            if doc_id not in doc_info:
                doc_info[doc_id] = item
    
    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [{'doc_id': d, 'rrf_score': s, 'rank': i+1, **doc_info[d]}
            for i, (d, s) in enumerate(ranked)]

# Compare different weight combinations
q = "Show me the performance chart"
text_r = text_retrieve(q, k=5)
vision_r = vision_retrieve(q, k=5)

for tw, vw in [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.3, 0.7), (0.0, 1.0)]:
    fused = weighted_rrf([text_r, vision_r], [tw, vw])
    ids = [x['doc_id'] for x in fused[:3]]
    print(f"  text={tw:.1f} vision={vw:.1f} → {ids}")


## Fallback Strategies

Not every query benefits from hybrid retrieval. A robust system needs fallback strategies:

1. **Modality confidence**: If the vision branch returns low-confidence results, fall back to text-only
2. **Query classification**: Detect if a query is visual ("show me the chart") or textual ("explain the algorithm")
3. **Graceful degradation**: If ColPali is down or slow, the text branch still works
4. **Threshold-based filtering**: Only include vision results above a confidence threshold

The goal is a system that's **always at least as good as text-only**, and better when vision helps.


In [ ]:
def smart_hybrid_retrieve(
    query: str,
    k: int = 5,
    vision_threshold: float = 0.1,
    visual_keywords: set = {'chart', 'diagram', 'image', 'show', 'visual', 'graph', 'table', 'figure'}
) -> dict:
    """Smart hybrid retrieval with fallback."""
    text_results = text_retrieve(query, k=k)
    
    # Check if query is likely visual
    query_words = set(query.lower().split())
    is_visual_query = bool(query_words & visual_keywords)
    
    if not is_visual_query:
        return {'mode': 'text_only', 'results': text_results, 'reason': 'Non-visual query'}
    
    vision_results = vision_retrieve(query, k=k)
    
    # Check vision confidence
    max_vision_score = max(r['score'] for r in vision_results) if vision_results else 0
    if max_vision_score < vision_threshold:
        return {'mode': 'text_only', 'results': text_results, 'reason': 'Low vision confidence'}
    
    # Full hybrid
    fused = reciprocal_rank_fusion(text_results, vision_results)
    return {'mode': 'hybrid', 'results': fused[:k], 'reason': 'Visual query + good vision signal'}

# Test smart routing
for q in ["explain the transformer", "show me the revenue chart", "what is the latency?"]:
    r = smart_hybrid_retrieve(q)
    print(f"  '{q}' → mode={r['mode']} ({r['reason']})")


## Exercises

### Exercise 1: Three-Way Fusion
Add a **BM25 (sparse) retriever** as a third branch alongside dense text and vision. Implement three-way RRF fusion and compare Recall@3 against two-way hybrid.

### Exercise 2: Adaptive Weights
Instead of fixed weights, implement a simple classifier that predicts the optimal text/vision weight ratio based on query characteristics (length, keywords, question type).

### Exercise 3: Real ColPali Integration
Replace the simulated vision embeddings with actual ColPali embeddings from notebook 10. Render your documents as page images, encode with ColPali, and measure the improvement.


## Key Takeaways

- **Text and vision retrieval are complementary** — text excels at semantic matching, vision at layout understanding
- **RRF is a robust fusion method** that requires no training and works across any pair of rankers
- **Weighted RRF** lets you tune the balance between modalities based on your query distribution
- **Smart routing** avoids unnecessary vision retrieval for purely textual queries
- **Always design for graceful degradation** — hybrid should never be worse than single-modality


## References

- Cormack, G. V., Clarke, C. L., & Buettcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods*. SIGIR.
- Faysse, M., et al. (2024). *ColPali: Efficient Document Retrieval with Vision Language Models*.
- Karpukhin, V., et al. (2020). *Dense Passage Retrieval for Open-Domain QA*. EMNLP.
- Lin, J. (2021). *A Proposed Conceptual Framework for a Representational Approach to Information Retrieval*.


---

[← 10 Page-as-Image Retrieval with ColPali](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/10_page_as_image_retrieval_with_colpali.ipynb) | [12 Multimodal Grounding & Evaluation →](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/12_multimodal_grounding_and_evaluation.ipynb)
